# fastLP synthetic benchmark

This notebook generates balanced synthetic panels, estimates local projections with `fastlp`, and reports wall-clock timing. The large benchmark has at least 10 million observations and is disabled by default because it requires several gigabytes of RAM during fixed-effect absorption.

In [5]:
from __future__ import annotations

from time import perf_counter
import gc

import numpy as np
import pandas as pd

from fastlp import LocalProjection

SEED = 2026
RUN_LARGE_BENCHMARK = True  # Set True only on a machine with ample RAM.

# 1,250,000 units x 8 periods = exactly 10,000,000 observations.
LARGE_UNITS = 1_250_000
LARGE_PERIODS = 8
assert LARGE_UNITS * LARGE_PERIODS >= 10_000_000

## Data generator

The outcome is generated from an anchor-time shock, a control, unit effects, time effects, and noise. Every unit has the same ordered time labels, satisfying fastLP v0.1's balanced-panel requirement.

In [2]:
def make_balanced_panel(n_units: int, n_periods: int, seed: int) -> pd.DataFrame:
    if n_units < 2 or n_periods < 3:
        raise ValueError('n_units must be >= 2 and n_periods must be >= 3')

    rng = np.random.default_rng(seed)
    unit = np.repeat(np.arange(n_units, dtype=np.int32), n_periods)
    time = np.tile(np.arange(n_periods, dtype=np.int16), n_units)
    unit_effect = rng.normal(scale=0.75, size=n_units).astype(np.float64)
    time_effect = rng.normal(scale=0.25, size=n_periods).astype(np.float64)
    shock = rng.normal(size=n_units * n_periods)
    control = rng.normal(size=n_units * n_periods)
    noise = rng.normal(scale=0.5, size=n_units * n_periods)
    outcome = 1.0 + 0.8 * shock - 0.35 * control + unit_effect[unit] + time_effect[time] + noise

    return pd.DataFrame({
        'unit': unit,
        'time': time,
        'outcome': outcome,
        'shock': shock,
        'control': control,
    })


def benchmark(data: pd.DataFrame, *, horizons: int, label: str) -> tuple[LocalProjection, pd.DataFrame]:
    start = perf_counter()
    estimator = LocalProjection(horizons=horizons, covariance='cluster').fit(
        data,
        outcome='outcome',
        shock='shock',
        controls=['control'],
        unit='unit',
        time='time',
        fixed_effects=['unit', 'time'],
        cluster='unit',
    )
    elapsed = perf_counter() - start
    result = estimator.to_frame().query("coefficient == 'shock'").copy()
    result.insert(0, 'benchmark', label)
    result['fit_seconds'] = elapsed
    result['n_input_rows'] = len(data)
    result['n_anchor_rows'] = estimator.n_obs_
    result['demean_backend'] = estimator.demeaning_diagnostics_['backend']
    return estimator, result


## Small benchmark

This run is suitable for a normal laptop and verifies the full estimation path before running the large case.

In [3]:
small_start = perf_counter()
small_data = make_balanced_panel(n_units=200, n_periods=40, seed=SEED)
small_generation_seconds = perf_counter() - small_start
small_model, small_results = benchmark(small_data, horizons=12, label='small')

print(f'Small data generation: {small_generation_seconds:.3f} seconds')
print(small_model.summary())
small_results

Small data generation: 0.011 seconds
LocalProjection(horizons=0..12, n_obs=5600, covariance=cluster, backend=rust)


,benchmark,horizon,coefficient,estimate,std_error,ci_low,ci_high,fit_seconds,n_input_rows,n_anchor_rows,demean_backend
0,small,0,shock,0.790253,0.006831,0.776863,0.803642,0.061133,8000,5600,rust
2,small,1,shock,-0.009168,0.012524,-0.033714,0.015377,0.061133,8000,5600,rust
4,small,2,shock,-0.065371,0.012666,-0.090195,-0.040547,0.061133,8000,5600,rust
6,small,3,shock,-0.045220,0.013049,-0.070796,-0.019644,0.061133,8000,5600,rust
8,small,4,shock,-0.030937,0.013933,-0.058245,-0.003628,0.061133,8000,5600,rust
10,small,5,shock,-0.006125,0.012600,-0.030821,0.018571,0.061133,8000,5600,rust
12,small,6,shock,0.007541,0.012526,-0.017011,0.032092,0.061133,8000,5600,rust
14,small,7,shock,-0.027839,0.013023,-0.053364,-0.002313,0.061133,8000,5600,rust
16,small,8,shock,-0.017612,0.013864,-0.044785,0.009561,0.061133,8000,5600,rust
18,small,9,shock,-0.020023,0.014677,-0.048789,0.008744,0.061133,8000,5600,rust


## 10 million-observation benchmark

The configuration below constructs exactly 10 million rows. It uses two horizons to keep the horizon-outcome matrix bounded while still exercising the shared-design cache. Enable `RUN_LARGE_BENCHMARK` above to generate the dataset, fit the estimator, and record the timings.

In [6]:
if RUN_LARGE_BENCHMARK:
    large_start = perf_counter()
    large_data = make_balanced_panel(
        n_units=LARGE_UNITS, n_periods=LARGE_PERIODS, seed=SEED + 1
    )
    large_generation_seconds = perf_counter() - large_start
    large_model, large_results = benchmark(large_data, horizons=2, label='large_10m')
    print(f'Large data generation: {large_generation_seconds:.3f} seconds')
    print(large_model.summary())
    display(large_results)
else:
    print('Large benchmark skipped. Set RUN_LARGE_BENCHMARK = True to run it.')

Large data generation: 0.744 seconds
LocalProjection(horizons=0..2, n_obs=7500000, covariance=cluster, backend=rust)


,benchmark,horizon,coefficient,estimate,std_error,ci_low,ci_high,fit_seconds,n_input_rows,n_anchor_rows,demean_backend
0,large_10m,0,shock,0.799724,0.000200,0.799331,0.800116,130.696957,10000000,7500000,rust
2,large_10m,1,shock,-0.133780,0.000378,-0.134521,-0.133039,130.696957,10000000,7500000,rust
4,large_10m,2,shock,-0.106925,0.000392,-0.107693,-0.106157,130.696957,10000000,7500000,rust


## Timing notes

Compare `fit_seconds` across runs, not just total notebook execution time. The estimator's speedup comes from residualizing the RHS and factorizing its Gram matrix once; outcome residualization and clustered covariance remain horizon-specific. For reproducible comparisons, record CPU model, RAM, BLAS thread settings, Python version, and whether the reported backend is `rust` or `numpy`.